In [4]:
from google.colab import files
files.upload()

Saving course.csv to course.csv


{'course.csv': b'course_id,course_name\r\n201,Introduction to Python\r\n202,Data Science Basics\r\n203,Machine Learning'}

In [5]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName('Online-Course-enrollment').getOrCreate()

In [6]:
student=spark.read.csv("student.csv",header=True,inferSchema=True)
student.show()

+----------+---------------+-----------+----------+---------+--------------+
|student_id|   student_name|blood_group|  phone_no|     city|student_status|
+----------+---------------+-----------+----------+---------+--------------+
|       101|    Alice Smith|         O+|9876543210|  Chennai|        Active|
|       102|      Bob Jones|         A+|9876543211|   Mumbai|        Active|
|       103|           NULL|         B+|9876543212|Bangalore|        Active|
|       104|  Charlie Brown|         O-|9876543213|    Delhi|        Active|
|       105|   Diana Prince|        AB+|9876543214|Hyderabad|        Active|
|       106|    Evan Wright|         A-|9876543215|    Kochi|        Active|
|       107|Fiona Gallagher|         O+|9876543216|     Pune|        Active|
|       108|   George Clark|         B-|9876543217|  Chennai|        Active|
|       109|  Hannah Abbott|         A+|9876543218|   Mumbai|        Active|
|       110|    Ian Malcolm|        AB-|9876543219|Bangalore|        Active|

In [7]:
course=spark.read.csv("course.csv",header=True,inferSchema=True)
course.show()

+---------+--------------------+
|course_id|         course_name|
+---------+--------------------+
|      201|Introduction to P...|
|      202| Data Science Basics|
|      203|    Machine Learning|
+---------+--------------------+



In [8]:
enrollment=spark.read.csv("enrollment.csv",header=True,inferSchema=True)
enrollment.show()

+-------------+----------+---------+---------------+-------------------+-----------------+
|enrollment_id|student_id|course_id|enrollment_date|progress_percentage|enrollment_status|
+-------------+----------+---------+---------------+-------------------+-----------------+
|         1001|       101|      201|     2026-01-15|                 85|         Enrolled|
|         1002|       102|      202|     2026-01-20|                 45|         Enrolled|
|         1003|       103|      201|     2026-02-01|                100|        Completed|
|         1004|       104|      203|     2026-01-10|                 12|          Dropped|
|         1005|       105|      202|           NULL|                105|         Enrolled|
|         1006|       106|      201|     2026-02-15|               NULL|         Enrolled|
|         1007|       107|      203|     2026-01-22|                100|        Completed|
|         1008|       108|      202|     2026-02-05|                 -5|          Dropped|

In [9]:
joined_df=student.join(enrollment,student.student_id==enrollment.student_id,"inner").join(course,enrollment.course_id==course.course_id,"inner")
joined_df.show()

+----------+---------------+-----------+----------+---------+--------------+-------------+----------+---------+---------------+-------------------+-----------------+---------+--------------------+
|student_id|   student_name|blood_group|  phone_no|     city|student_status|enrollment_id|student_id|course_id|enrollment_date|progress_percentage|enrollment_status|course_id|         course_name|
+----------+---------------+-----------+----------+---------+--------------+-------------+----------+---------+---------------+-------------------+-----------------+---------+--------------------+
|       101|    Alice Smith|         O+|9876543210|  Chennai|        Active|         1001|       101|      201|     2026-01-15|                 85|         Enrolled|      201|Introduction to P...|
|       102|      Bob Jones|         A+|9876543211|   Mumbai|        Active|         1002|       102|      202|     2026-01-20|                 45|         Enrolled|      202| Data Science Basics|
|       103|   

In [11]:
from pyspark.sql.functions import col, count, sum, when
print("Displaying top courses and dropout courses")
course_summary = joined_df.groupBy(course["course_id"], "course_name").agg(
    count("enrollment_id").alias("Total_Enrolled"),
    sum(when(col("enrollment_status") == "Completed", 1).otherwise(0)).alias("Total_Completed"),
    sum(when(col("enrollment_status") == "Dropped", 1).otherwise(0)).alias("Total_Dropouts")
)

print("--- Output showing top completed and dropped-out courses ---")
course_summary.orderBy(col("Total_Completed").desc(), col("Total_Dropouts").desc()).show()

Displaying top courses and dropout courses
--- Output showing top completed and dropped-out courses ---
+---------+--------------------+--------------+---------------+--------------+
|course_id|         course_name|Total_Enrolled|Total_Completed|Total_Dropouts|
+---------+--------------------+--------------+---------------+--------------+
|      203|    Machine Learning|             3|              1|             1|
|      201|Introduction to P...|             4|              1|             0|
|      202| Data Science Basics|             3|              0|             1|
+---------+--------------------+--------------+---------------+--------------+

